In [ ]:
!pip install -q streamlit
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 63.6 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 2s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [ ]:
import zipfile
import os

with zipfile.ZipFile("archive.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset unzipped successfully!")

Dataset unzipped successfully!


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

DATA_DIR = "dataset/archive/dataset-resized"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=123, image_size=(224, 224), batch_size=32
)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=123, image_size=(224, 224), batch_size=32
)

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(6, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_dataset, validation_data=validation_dataset, epochs=5)
model.save('waste_classifier.h5')

Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 63s 567ms/step - accuracy: 0.6296 - loss: 1.0535 - val_accuracy: 0.8099 - val_loss: 0.6186
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 55ms/step - accuracy: 0.8289 - loss: 0.5417 - val_accuracy: 0.8554 - val_loss: 0.4818
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.8665 - loss: 0.4348 - val_accuracy: 0.8772 - val_loss: 0.4242
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.8783 - loss: 0.3730 - val_accuracy: 0.8871 - val_loss: 0.3926
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.8971 - loss: 0.3252 - val_accuracy: 0.8851 - val_loss: 0.3679


In [ ]:
%%writefile gradcam.py
import numpy as np
import tensorflow as tf
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    base_model = model.layers[0]
    classifier_layers = model.layers[1:]

    with tf.GradientTape() as tape:
        conv_outputs = base_model(img_array, training=False)
        tape.watch(conv_outputs)

        x = conv_outputs
        for layer in classifier_layers:
            if isinstance(layer, tf.keras.layers.Dropout):
                x = layer(x, training=False)
            else:
                x = layer(x)

        preds = x

        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = heatmap * alpha + img
    return np.clip(superimposed_img, 0, 255).astype('uint8')

Writing gradcam.py


In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2
from gradcam import make_gradcam_heatmap, overlay_heatmap

st.set_page_config(page_title="Waste Classifier", layout="wide")

@st.cache_resource
def load_model():
    return tf.keras.models.load_model('waste_classifier.h5')

model = load_model()
CLASSES = ['Cardboard', 'Glass', 'Metal', 'Paper', 'Plastic', 'Trash']

st.title("♻️ AI Waste Sorting System")
uploaded_file = st.file_uploader("Upload an image", type=["jpg", "png", "jpeg"])

if uploaded_file:
    with open("temp.jpg", "wb") as f:
        f.write(uploaded_file.getbuffer())

    image = Image.open(uploaded_file).convert('RGB')
    img_array = tf.expand_dims(tf.keras.preprocessing.image.img_to_array(image.resize((224, 224))), 0)

    with st.spinner("Analyzing..."):
        predictions = model.predict(img_array)
        score = tf.nn.softmax(predictions[0])
        st.success(f"**Prediction:** {CLASSES[np.argmax(score)]} ({100 * np.max(score):.1f}%)")

        cam_image = cv2.cvtColor(overlay_heatmap("temp.jpg", make_gradcam_heatmap(img_array, model, 'top_activation')), cv2.COLOR_BGR2RGB)

    c1, c2 = st.columns(2)
    c1.image(image, caption="Original", use_container_width=True)
    c2.image(cam_image, caption="Grad-CAM", use_container_width=True)

Writing app.py
